# 🧠 Иерархический поведенческий анализ пользователей с Z-score Отклонениями и Causal XAI (v10)

**Подход:** HST-UEBA (Иерархический Sequential Transformer c микро-признаками времени и поведенческими Z-отклонениями) на скользящих окнах (Sliding Window).

Модель анализирует траекторию поведения пользователя за окно в 7 дней и классифицирует наличие аномалий в поведении на **последний день окна** (контекстное обнаружение аномалий - Contextual Detection).

**Улучшения в v10:**
* **Z-score нормализация отклонений (User Behavioral Baselines):** Расчет суточных профилей (среднее и std событий, USB, email, HTTP) на обучающей выборке и z-score отклонения признаков с адаптивным fallback к global_std при малой дисперсии.
* **Устранение Target Leakage в обрезке:** Независимая от разметки эвристическая обрезка (Suspicious Trimming) по априорным весам подозрительных токенов.
* **Временная непрерывность окон:** Исключение окон с временными разрывами активности более 3 дней.
* **Расширенный словарь токенов (Extended Vocabulary):** Расширение с 19 до 27 токенов для детального учета типов USB-копирования (архивы/exe), внеурочного времени входа и рассылки bulk/large email.
* **Устранение конфликтов в Attentive Pooling:** Расчет внимания по контексту дней 1-6 и интерполяция с целевым днем 7.
* **Исправление hidden states BiLSTM:** Корректное извлечение выходов последнего слоя при `num_lstm_layers=2`.
* **Оценка безопасности (SOC metrics):** Интеграция MTTD (Mean Time To Detect) и метрик Recall@1%FPR для объективной оценки модели.
* **Абляция с изолированной валидацией:** Корректный Ablation Study на непересекающихся пользователях.
* **Multiple Seeds:** Обучение модели на 3 случайных семенах с выводом `mean ± std` и диапазона `min-max` для надежности.

In [ ]:
import numpy as np
import json
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_fscore_support,
    roc_curve, auc, precision_recall_curve, fbeta_score, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import csv
import warnings
warnings.filterwarnings('ignore')

# Проверяем GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используемое устройство: {device}')

## 1. Загрузка данных, разделение пользователей и проверка согласованности

In [ ]:
DATA_PATH = 'cert_nlp_sequences.npz'
# Загружаем сырые данные
data = np.load(DATA_PATH, allow_pickle=True)
X = data['X']
X_hours = data.get('X_hours', np.zeros_like(X))
X_working_hours = data.get('X_working_hours', np.zeros_like(X))
X_dev = data['X_dev']
y = data['y']
users = data['users']
dates = data['dates']
metadata = json.loads(str(data['metadata']))

users_dev = data.get('users_dev', users)
dates_dev = data.get('dates_dev', dates)

# Строгие проверки согласованности порядка строк препроцессинга
assert len(X_dev) == len(X), 'Mismatch between X and X_dev length'
assert np.all(users == users_dev), 'User order mismatch between X and X_dev'
assert np.all(dates == dates_dev), 'Date order mismatch between X and X_dev'

TOKEN_MAP = metadata['token_map']
VOCAB_SIZE = metadata['vocab_size']
MAX_SEQ_LEN = metadata['max_seq_len']
DEV_GLOBAL_MEAN = np.array(metadata.get('global_mean', [0.0, 0.0, 0.0, 0.0]))
DEV_GLOBAL_STD = np.array(metadata.get('global_std', [1.0, 1.0, 1.0, 1.0]))

# 1. Загружаем готовое разбиение пользователей
if 'train_users' in data:
    train_users = data['train_users']
    val_users = data['val_users']
    test_users = data['test_users']
else:
    unique_users = np.unique(users)
    np.random.seed(42)
    shuffled_users = unique_users.copy()
    np.random.shuffle(shuffled_users)
    n_users = len(shuffled_users)
    train_split = int(0.7 * n_users)
    val_split = int(0.85 * n_users)
    train_users = shuffled_users[:train_split]
    val_users = shuffled_users[train_split:val_split]
    test_users = shuffled_users[val_split:]

print(f'Распределение пользователей - Train: {len(train_users)}, Val: {len(val_users)}, Test: {len(test_users)}')

# 2. Вычисляем Z-нормализацию tsle строго на днях пользователей из Train с исключением padding
X_tsle_raw = data.get('X_tsle', np.zeros_like(X, dtype=np.float32))
X_tsle_log = np.log1p(X_tsle_raw)

train_day_mask = np.isin(users, train_users)
nonpad_mask = (X != 0)
tsle_mask = train_day_mask[:, None] & nonpad_mask
tsle_mean = X_tsle_log[tsle_mask].mean()
tsle_std = X_tsle_log[tsle_mask].std() + 1e-8

# Применяем нормализацию к полному массиву
X_tsle = (X_tsle_log - tsle_mean) / tsle_std
print(f'Параметры нормализации TSLE (по Train-дням) - Mean: {tsle_mean:.4f}, Std: {tsle_std:.4f}')
print(f'Загружено {len(X)} дней активности')

## 2. Формирование скользящих окон (Sliding Window) и анализ Target Leakage

> [!NOTE]
> **Целевая постановка задачи:** В системах класса UEBA мы выполняем **контекстное обнаружение аномалий (Contextual Anomaly Detection)**. > Входные данные окна `[Day 1, Day 2, ..., Day 7]` используются для классификации состояния `Day 7` (атака или норма). Текущая активность пользователя (`Day 7`) оценивается в контексте его недавней предыстории за предыдущие 6 дней.
> Это отличается от задачи прогнозирования (Forecasting), где по `[Day 1..7]` предсказывался бы `Day 8`. Схема `[Day 1..7] -> Label(Day 7)` является стандартной и математически корректной для систем мониторинга инцидентов ИБ в реальном времени.

In [ ]:
WINDOW_SIZE = 7

df = pd.DataFrame({
    'user': users,
    'date': dates,
    'y': y,
    'idx': np.arange(len(y))
})
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.dayofweek
df = df.sort_values(by=['user', 'date'])

win_X, win_h, win_wh, win_tsle, win_dev, win_y, win_u, win_dow, win_dates = [], [], [], [], [], [], [], [], []

for user, group in df.groupby('user'):
    indices = group['idx'].values
    labels = group['y'].values
    dows = group['day_of_week'].values
    g_dates = group['date'].values
    
    if len(indices) < WINDOW_SIZE:
        continue  # Пропускаем пользователей с короткой историей
    
    for i in range(len(indices) - WINDOW_SIZE + 1):
        w_dates = g_dates[i:i+WINDOW_SIZE]
        # Проверяем временную непрерывность траектории (пропуск более 3 дней — отбрасываем окно)
        date_diffs = np.diff(w_dates).astype('timedelta64[D]').astype(int)
        if len(date_diffs) > 0 and np.max(date_diffs) > 3:
            continue
            
        w_idx = indices[i:i+WINDOW_SIZE]
        w_labels = labels[i:i+WINDOW_SIZE]
        w_dows = dows[i:i+WINDOW_SIZE]
        
        win_X.append(X[w_idx])
        win_h.append(X_hours[w_idx])
        win_wh.append(X_working_hours[w_idx])
        win_tsle.append(X_tsle[w_idx])
        win_dev.append(X_dev[w_idx])
        win_y.append(int(w_labels[-1]))  # Предсказание на последний день окна
        win_u.append(user)
        win_dow.append(w_dows)
        win_dates.append(w_dates)

win_X = np.array(win_X)
win_h = np.array(win_h)
win_wh = np.array(win_wh)
win_tsle = np.array(win_tsle)
win_dev = np.array(win_dev)
win_y = np.array(win_y)
win_u = np.array(win_u)
win_dow = np.array(win_dow)
win_dates = np.array(win_dates)

print(f'Сформировано окон: {len(win_X)}')
print(f'Окон с атакми: {win_y.sum()}')

## 3. Подготовка PyTorch Dataset и DataLoader по группам пользователей

In [ ]:
train_idx = np.where(np.isin(win_u, train_users))[0]
val_idx = np.where(np.isin(win_u, val_users))[0]
test_idx = np.where(np.isin(win_u, test_users))[0]

class HierarchicalDataset(Dataset):
    def __init__(self, X, X_h, X_wh, X_tsle, dow, X_dev, y):
        self.X = torch.LongTensor(X)
        self.X_h = torch.LongTensor(X_h)
        self.X_wh = torch.FloatTensor(X_wh)
        self.X_tsle = torch.FloatTensor(X_tsle)
        self.dow = torch.LongTensor(dow)
        self.X_dev = torch.FloatTensor(X_dev)
        self.y = torch.FloatTensor(y)
        
    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return self.X[idx], self.X_h[idx], self.X_wh[idx], self.X_tsle[idx], self.dow[idx], self.X_dev[idx], self.y[idx]

BATCH_SIZE = 64
train_loader = DataLoader(
    HierarchicalDataset(win_X[train_idx], win_h[train_idx], win_wh[train_idx], win_tsle[train_idx], win_dow[train_idx], win_dev[train_idx], win_y[train_idx]), 
    batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(
    HierarchicalDataset(win_X[val_idx], win_h[val_idx], win_wh[val_idx], win_tsle[val_idx], win_dow[val_idx], win_dev[val_idx], win_y[val_idx]), 
    batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(
    HierarchicalDataset(win_X[test_idx], win_h[test_idx], win_wh[test_idx], win_tsle[test_idx], win_dow[test_idx], win_dev[test_idx], win_y[test_idx]), 
    batch_size=BATCH_SIZE, shuffle=False)

print(f'Train size: {len(train_idx)} | Val size: {len(val_idx)} | Test size: {len(test_idx)}')

## 4. Архитектура модели HST-UEBA v8 (с Z-отклонениями, LayerNorm, Sigmoid Pooling и GELU Classifier) & Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.85, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

class AttentivePooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)
        # Инициализируем вес последнего дня через логит от 0.3
        self.last_day_weight = nn.Parameter(torch.logit(torch.tensor(0.3)))
        
    def forward(self, trans_out):
        # trans_out has shape (B, W, hidden_dim)
        # Вычисляем внимание только по контекстным дням (1-6) во избежание дублирования
        context_days = trans_out[:, :-1, :]
        attn_scores = torch.softmax(self.attention(context_days).squeeze(-1), dim=1)
        weighted = (context_days * attn_scores.unsqueeze(-1)).sum(dim=1)
        # Явно выделяем последний день скользящего окна с Sigmoid
        last_day = trans_out[:, -1, :]
        w = torch.sigmoid(self.last_day_weight)
        return (1.0 - w) * weighted + w * last_day

class HierarchicalTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, window_size=7, num_heads=4, num_layers=2, num_lstm_layers=2):
        super().__init__()
        # Уровень 1: День
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.hour_embedding = nn.Embedding(24, embed_dim // 4)
        
        lstm_input_dim = embed_dim + (embed_dim // 4) + 2
        self.day_lstm = nn.LSTM(lstm_input_dim, hidden_dim//2, num_layers=num_lstm_layers, batch_first=True, bidirectional=True, dropout=0.2 if num_lstm_layers > 1 else 0.0)
        
        # Слой слияния суточных эмбеддингов с Z-отклонениями (4 признака)
        self.day_proj = nn.Linear(hidden_dim + 4, hidden_dim)
        
        # Уровень 2: Окно (Траектория)
        self.dow_embedding = nn.Embedding(7, hidden_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, window_size, hidden_dim) * 0.02)
        
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads, batch_first=True, dropout=0.3)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Attentive Pooling (v9)
        self.pool = AttentivePooling(hidden_dim)
        
        # Простой классификатор с GELU
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
        
    def forward(self, x, x_h, x_wh, x_tsle, dow, x_dev, extract_attention=False):
        B, W, L = x.shape
        x = x.view(B * W, L)
        x_h = x_h.view(B * W, L)
        x_wh = x_wh.view(B * W, L, 1)
        x_tsle = x_tsle.view(B * W, L, 1)
        
        emb = self.embedding(x)
        h_emb = self.hour_embedding(x_h)
        
        lstm_in = torch.cat([emb, h_emb, x_wh, x_tsle], dim=-1)
        
        # Вычисляем длины последовательностей для pack_padded_sequence
        lengths = (x != 0).sum(dim=1).cpu()
        lengths = torch.clamp(lengths, min=1)
        
        packed = nn.utils.rnn.pack_padded_sequence(lstm_in, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.day_lstm(packed)
        
        # Явно берем последний слой двунаправленного LSTM (индексы -2 и -1)
        day_repr = torch.cat([h_n[-2], h_n[-1]], dim=1)
        # Конкатенируем Z-отклонения
        dev_features = x_dev.view(B * W, 4)
        day_repr = torch.cat([day_repr, dev_features], dim=-1)
        day_repr = self.day_proj(day_repr)
        
        seq_repr = day_repr.view(B, W, -1)
        seq_repr = seq_repr + self.dow_embedding(dow) + self.pos_encoder
        seq_repr = self.layer_norm(seq_repr)
        
        # Вычисляем маску пустых дней на уровне окон
        day_mask = (x.view(B, W, L).sum(dim=-1) == 0)
        
        if extract_attention:
            out = seq_repr
            for layer in self.transformer.layers[:-1]:
                out = layer(out, src_key_padding_mask=day_mask)
                
            last_layer = self.transformer.layers[-1]
            if getattr(last_layer, 'norm_first', False):
                norm_out = last_layer.norm1(out)
                attn_out, attn_weights = last_layer.self_attn(
                    norm_out, norm_out, norm_out, 
                    key_padding_mask=day_mask, 
                    average_attn_weights=True
                )
                x_attn = out + last_layer.dropout1(attn_out)
                norm_ff = last_layer.norm2(x_attn)
                ff_out = last_layer.linear2(last_layer.dropout(last_layer.activation(last_layer.linear1(norm_ff))))
                trans_out = x_attn + last_layer.dropout2(ff_out)
            else:
                attn_out, attn_weights = last_layer.self_attn(
                    out, out, out, 
                    key_padding_mask=day_mask, 
                    average_attn_weights=True
                )
                x_attn = out + last_layer.dropout1(attn_out)
                x_norm1 = last_layer.norm1(x_attn)
                ff_out = last_layer.linear2(last_layer.dropout(last_layer.activation(last_layer.linear1(x_norm1))))
                x_ff = x_norm1 + last_layer.dropout2(ff_out)
                trans_out = last_layer.norm2(x_ff)
        else:
            trans_out = self.transformer(seq_repr, src_key_padding_mask=day_mask)
            attn_weights = None
            
        pooled = self.pool(trans_out)
        logits = self.classifier(pooled)
        return logits.squeeze(-1), trans_out, attn_weights

## 4.1. Абляционный анализ (Ablation Study) на подмножестве пользователей
Для подтверждения вклада каждого изменения в итоговое качество системы, мы проведем быстрое обучение (по 3 эпохи) для 5 различных конфигураций модели на небольшом подмножестве обучающих пользователей.

In [ ]:
ablation_train_users = train_users[:12]
ablation_val_users = train_users[12:15]
ablation_train_idx = np.where(np.isin(win_u, ablation_train_users))[0]
ablation_val_idx = np.where(np.isin(win_u, ablation_val_users))[0]

ablation_loader = DataLoader(
    HierarchicalDataset(
        win_X[ablation_train_idx], win_h[ablation_train_idx], win_wh[ablation_train_idx], 
        win_tsle[ablation_train_idx], win_dow[ablation_train_idx], win_dev[ablation_train_idx], 
        win_y[ablation_train_idx]
    ), 
    batch_size=BATCH_SIZE, shuffle=True
)
ablation_val_loader = DataLoader(
    HierarchicalDataset(
        win_X[ablation_val_idx], win_h[ablation_val_idx], win_wh[ablation_val_idx], 
        win_tsle[ablation_val_idx], win_dow[ablation_val_idx], win_dev[ablation_val_idx], 
        win_y[ablation_val_idx]
    ), 
    batch_size=BATCH_SIZE, shuffle=False
)

configs = [
    {'name': 'v7 baseline', 'layer_norm': False, 'gelu': False, 'num_lstm_layers': 1, 'sigmoid_w': False, 'use_dev': False},
    {'name': '+ X_dev', 'layer_norm': False, 'gelu': False, 'num_lstm_layers': 1, 'sigmoid_w': False, 'use_dev': True},
    {'name': '+ LayerNorm', 'layer_norm': True, 'gelu': False, 'num_lstm_layers': 1, 'sigmoid_w': False, 'use_dev': True},
    {'name': '+ GELU head', 'layer_norm': True, 'gelu': True, 'num_lstm_layers': 1, 'sigmoid_w': False, 'use_dev': True},
    {'name': '+ 2-layer LSTM', 'layer_norm': True, 'gelu': True, 'num_lstm_layers': 2, 'sigmoid_w': False, 'use_dev': True},
    {'name': 'Full v10 (+Sigmoid w)', 'layer_norm': True, 'gelu': True, 'num_lstm_layers': 2, 'sigmoid_w': True, 'use_dev': True}
]

ablation_results = []
base_pr_auc, base_f2 = 0.0, 0.0

for idx, cfg in enumerate(configs):
    print(f'Обучение конфигурации {idx+1}/{len(configs)}: {cfg["name"]}...')
    
    class AblationModel(nn.Module):
        def __init__(self, use_ln, use_gelu, lstm_layers, use_sigmoid_w, use_dev):
            super().__init__()
            self.use_ln = use_ln
            self.use_gelu = use_gelu
            self.use_sigmoid_w = use_sigmoid_w
            self.use_dev = use_dev
            
            self.embedding = nn.Embedding(VOCAB_SIZE, 64, padding_idx=0)
            self.hour_embedding = nn.Embedding(24, 16)
            
            lstm_input_dim = 64 + 16 + 2
            self.day_lstm = nn.LSTM(lstm_input_dim, 64, num_layers=lstm_layers, batch_first=True, bidirectional=True, dropout=0.2 if lstm_layers > 1 else 0.0)
            
            if use_dev:
                self.day_proj = nn.Linear(128 + 4, 128)
            else:
                self.day_proj = nn.Linear(128, 128)
                
            self.dow_embedding = nn.Embedding(7, 128)
            self.pos_encoder = nn.Parameter(torch.randn(1, 7, 128) * 0.02)
            self.layer_norm = nn.LayerNorm(128)
            
            encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True, dropout=0.3)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
            self.pool_attn = nn.Linear(128, 1)
            
            if use_sigmoid_w:
                init_w = 0.3
                self.last_day_weight = nn.Parameter(torch.logit(torch.tensor(init_w)))
            else:
                self.last_day_weight = nn.Parameter(torch.tensor(0.0 if use_sigmoid_w else 0.3))
            
            if use_gelu:
                self.classifier = nn.Sequential(
                    nn.Linear(128, 64),
                    nn.GELU(),
                    nn.Dropout(0.3),
                    nn.Linear(64, 1)
                )
            else:
                self.classifier = nn.Sequential(
                    nn.Linear(128, 64),
                    nn.ReLU(),
                    nn.Dropout(0.3),
                    nn.Linear(64, 1)
                )
                
        def forward(self, x, x_h, x_wh, x_tsle, dow, x_dev):
            B, W, L = x.shape
            x_flat = x.view(B * W, L)
            emb = self.embedding(x_flat)
            h_emb = self.hour_embedding(x_h.view(B * W, L))
            lstm_in = torch.cat([emb, h_emb, x_wh.view(B * W, L, 1), x_tsle.view(B * W, L, 1)], dim=-1)
            
            lengths = torch.clamp((x_flat != 0).sum(dim=1).cpu(), min=1)
            packed = nn.utils.rnn.pack_padded_sequence(lstm_in, lengths, batch_first=True, enforce_sorted=False)
            _, (h_n, _) = self.day_lstm(packed)
            
            # Явно берем последний слой двунаправленного LSTM (индексы -2 и -1)
            day_repr = torch.cat([h_n[-2], h_n[-1]], dim=1)
                
            if self.use_dev:
                day_repr = torch.cat([day_repr, x_dev.view(B * W, 4)], dim=-1)
            day_repr = self.day_proj(day_repr)
            
            seq_repr = day_repr.view(B, W, -1) + self.dow_embedding(dow) + self.pos_encoder
            if self.use_ln:
                seq_repr = self.layer_norm(seq_repr)
                
            day_mask = (x.view(B, W, L).sum(dim=-1) == 0)
            trans_out = self.transformer(seq_repr, src_key_padding_mask=day_mask)
            
            # Вычисляем внимание только по контекстным дням траектории (1-6)
            context_days = trans_out[:, :-1, :]
            attn_scores = torch.softmax(self.pool_attn(context_days).squeeze(-1), dim=1)
            weighted = (context_days * attn_scores.unsqueeze(-1)).sum(dim=1)
            last_day = trans_out[:, -1, :]
            
            if self.use_sigmoid_w:
                w = torch.sigmoid(self.last_day_weight)
            else:
                w = self.last_day_weight
            pooled = (1.0 - w) * weighted + w * last_day
            
            return self.classifier(pooled).squeeze(-1)

    ab_model = AblationModel(cfg['layer_norm'], cfg['gelu'], cfg['num_lstm_layers'], cfg['sigmoid_w'], cfg['use_dev']).to(device)
    criterion = FocalLoss()
    optimizer = torch.optim.AdamW(ab_model.parameters(), lr=1e-3)
    
    for epoch in range(3):
        ab_model.train()
        for batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y in ablation_loader:
            batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y = \
                batch_X.to(device), batch_h.to(device), batch_wh.to(device), batch_tsle.to(device), batch_dow.to(device), batch_dev.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = ab_model(batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            
    ab_model.eval()
    val_probs, val_true = [], []
    with torch.no_grad():
        for batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y in ablation_val_loader:
            batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev = \
                batch_X.to(device), batch_h.to(device), batch_wh.to(device), batch_tsle.to(device), batch_dow.to(device), batch_dev.to(device)
            logits = ab_model(batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev)
            val_probs.extend(torch.sigmoid(logits).cpu().numpy())
            val_true.extend(batch_y.numpy())
            
    val_probs = np.array(val_probs)
    val_true = np.array(val_true)
    pr_auc = average_precision_score(val_true, val_probs)
    
    best_f2_ab = 0.0
    for t in np.linspace(0.01, 0.99, 50):
        preds = (val_probs >= t).astype(int)
        f2 = fbeta_score(val_true, preds, beta=2, zero_division=0)
        if f2 > best_f2_ab:
            best_f2_ab = f2
            
    if idx == 0:
        base_pr_auc = pr_auc
        base_f2 = best_f2_ab
        d_pr = '—'
        d_f2 = '—'
    else:
        d_pr = f'{((pr_auc - base_pr_auc)/base_pr_auc*100):+.2f}%' if base_pr_auc > 0 else '+0.00%'
        d_f2 = f'{((best_f2_ab - base_f2)/base_f2*100):+.2f}%' if base_f2 > 0 else '+0.00%'
        
    ablation_results.append({
        'Модель': cfg['name'],
        'PR-AUC': pr_auc,
        'F2': best_f2_ab,
        'Δ PR-AUC': d_pr,
        'Δ F2': d_f2
    })

df_ablation = pd.DataFrame(ablation_results)
print('\n' + '='*80)
print('📊 ТАБЛИЦА АБЛЯЦИОННОГО АНАЛИЗА (ABLATION STUDY)')
print('='*80)
print(df_ablation.to_markdown(index=False))
print('='*80)

## 5. Обучение финальной модели HST-UEBA v10 на 3 случайных семенах (Multiple Seeds)

In [ ]:
import copy

def split_stats(name, idx):
    labels = win_y[idx]
    print(f'{name}: windows={len(idx)}, positives={labels.sum()}, positive_rate={labels.mean():.6f}')

split_stats('Train', train_idx)
split_stats('Val', val_idx)
split_stats('Test', test_idx)

SEEDS = [42, 100, 2026]
NUM_EPOCHS = 15
PATIENCE = 5

run_metrics = []
best_overall_val_pr_auc = 0.0
best_model_state_global = None
best_history_global = {'train_loss': [0], 'val_loss': [0], 'val_pr_auc': [0]}

for run_idx, seed in enumerate(SEEDS):
    print(f'\nЗапуск обучения #{run_idx+1} (Seed={seed})...')
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    model = HierarchicalTransformer(vocab_size=VOCAB_SIZE, window_size=WINDOW_SIZE, num_lstm_layers=2).to(device)
    criterion = FocalLoss(alpha=0.85, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)
    
    best_pr_auc = 0.0
    best_model_state = None
    patience_counter = 0
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_pr_auc': []
    }
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y in train_loader:
            batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y = \
                batch_X.to(device), batch_h.to(device), batch_wh.to(device), batch_tsle.to(device), batch_dow.to(device), batch_dev.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits, _, _ = model(batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, extract_attention=False)
            loss = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * batch_X.size(0)
            
        epoch_train_loss = running_loss / len(train_idx)
        
        model.eval()
        val_preds, val_labels = [], []
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y in val_loader:
                batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y_gpu = \
                    batch_X.to(device), batch_h.to(device), batch_wh.to(device), batch_tsle.to(device), batch_dow.to(device), batch_dev.to(device), batch_y.to(device)
                logits, _, _ = model(batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, extract_attention=False)
                v_loss = criterion(logits, batch_y_gpu)
                val_loss += v_loss.item() * batch_X.size(0)
                probs = torch.sigmoid(logits)
                val_preds.extend(probs.cpu().numpy())
                val_labels.extend(batch_y.numpy())
                
        epoch_val_loss = val_loss / len(val_idx)
        pr_auc = average_precision_score(val_labels, val_preds)
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_pr_auc'].append(pr_auc)
        
        if pr_auc > best_pr_auc:
            best_pr_auc = pr_auc
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            marker = '*'
        else:
            patience_counter += 1
            marker = ''
            
        scheduler.step(pr_auc)
        print(f'Epoch {epoch+1:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val PR-AUC: {pr_auc:.4f} {marker}')
        
        if patience_counter >= PATIENCE:
            print('Early stopping triggered!')
            break
            
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        
    model.eval()
    test_probs, test_true = [], []
    with torch.no_grad():
        for batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y in test_loader:
            batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev = \
                batch_X.to(device), batch_h.to(device), batch_wh.to(device), batch_tsle.to(device), batch_dow.to(device), batch_dev.to(device)
            logits, _, _ = model(batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, extract_attention=False)
            test_probs.extend(torch.sigmoid(logits).cpu().numpy())
            test_true.extend(batch_y.numpy())
            
    test_probs = np.array(test_probs)
    test_true = np.array(test_true)
    
    val_probs = []
    with torch.no_grad():
        for batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y in val_loader:
            batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev = \
                batch_X.to(device), batch_h.to(device), batch_wh.to(device), batch_tsle.to(device), batch_dow.to(device), batch_dev.to(device)
            logits, _, _ = model(batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, extract_attention=False)
            val_probs.extend(torch.sigmoid(logits).cpu().numpy())
            
    best_f2 = 0.0
    best_thr = 0.5
    for t in np.linspace(0.01, 0.99, 100):
        preds = (np.array(val_probs) >= t).astype(int)
        f2 = fbeta_score(val_labels, preds, beta=2, zero_division=0)
        if f2 > best_f2:
            best_f2 = f2
            best_thr = t
            
    final_test_preds = (test_probs >= best_thr).astype(int)
    report = classification_report(test_true, final_test_preds, digits=4, output_dict=True)
    
    test_pr_auc = average_precision_score(test_true, test_probs)
    test_f2 = fbeta_score(test_true, final_test_preds, beta=2)
    test_roc_auc = auc(roc_curve(test_true, test_probs)[0], roc_curve(test_true, test_probs)[1])
    
    run_metrics.append({
        'seed': seed,
        'pr_auc': test_pr_auc,
        'f2_score': test_f2,
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'roc_auc': test_roc_auc
    })
    
    if best_pr_auc > best_overall_val_pr_auc:
        best_overall_val_pr_auc = best_pr_auc
        best_model_state_global = copy.deepcopy(best_model_state)
        best_thr_global = best_thr
        best_history_global = copy.deepcopy(history)
        best_test_probs_global = test_probs
        best_test_true_global = test_true

if best_model_state_global is not None:
    model.load_state_dict(best_model_state_global)
    best_thr = best_thr_global
    test_probs = best_test_probs_global
    test_true = best_test_true_global

## 6. Кривые обучения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(best_history_global['train_loss'], label='Train Focal Loss', lw=2)
axes[0].plot(best_history_global['val_loss'], label='Val Focal Loss', lw=2)
axes[0].set_title('Loss Curves (Best Run)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(best_history_global['val_pr_auc'], label='Val PR-AUC', color='green', lw=2)
axes[1].set_title('Validation PR-AUC (Best Run)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PR-AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.show()

## 7. Финальная оценка качества модели HST-UEBA и сравнение с бейслайнами CERT r5.2
1. Выводим средние метрики и стабильность по 3 случайным семенам.
2. Строим Confusion Matrix, ROC-кривую, PR-кривую для наилучшего запуска и выводим итоговую таблицу сравнения с бейслайнами.

In [ ]:
df_runs = pd.DataFrame(run_metrics)
print('=' * 80)
print('📊 СТАБИЛЬНОСТЬ МОДЕЛИ НА 3 СЛУЧАЙНЫХ СЕМЕНАХ (MULTIPLE SEEDS)')
print('=' * 80)
for col in ['roc_auc', 'pr_auc', 'precision', 'recall', 'f2_score']:
    mean_val = df_runs[col].mean()
    std_val = df_runs[col].std()
    min_val = df_runs[col].min()
    max_val = df_runs[col].max()
    print(f'{col:<12} : {mean_val:.4f} ± {std_val:.4f} (range: {min_val:.4f} - {max_val:.4f})')
print('=' * 80 + '\n')

fpr, tpr, thresholds = roc_curve(test_true, test_probs)
roc_auc = auc(fpr, tpr)
pr_auc = average_precision_score(test_true, test_probs)

target_fpr = 0.01
idx = np.where(fpr <= target_fpr)[0][-1]
recall_at_fixed_fpr = tpr[idx]
thr_at_fixed_fpr = thresholds[idx]

final_test_preds = (test_probs >= best_thr).astype(int)
report = classification_report(test_true, final_test_preds, digits=4, output_dict=True)
cm = confusion_matrix(test_true, final_test_preds)

# Функция расчета MTTD (Mean Time To Detect) в днях
def compute_mttd(test_true, test_probs, win_dates, win_u, test_idx, threshold):
    mttd_values = []
    unique_test_users = np.unique(win_u[test_idx])
    for user in unique_test_users:
        user_mask = (win_u[test_idx] == user)
        user_labels = test_true[user_mask]
        user_probs = test_probs[user_mask]
        user_dates = win_dates[test_idx][user_mask]
        
        if user_labels.sum() == 0:
            continue
        
        # Находим первый день атаки
        attack_start = None
        for label, dates in zip(user_labels, user_dates):
            if label == 1:
                attack_start = pd.to_datetime(dates[-1])
                break
        
        # Находим первое обнаружение
        for prob, dates in zip(user_probs, user_dates):
            if prob >= threshold:
                detection_time = pd.to_datetime(dates[-1])
                if attack_start and detection_time >= attack_start:
                    mttd_values.append((detection_time - attack_start).days)
                    break
    return np.mean(mttd_values) if mttd_values else float('inf')

mttd_value = compute_mttd(test_true, test_probs, win_dates, win_u, test_idx, best_thr)

# Строим красивую визуализацию
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# 1. Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            annot_kws={'size': 14, 'weight': 'bold'},
            xticklabels=['Норма', 'Угроза'], yticklabels=['Норма', 'Угроза'])
axes[0].set_title('Матрица ошибок (Confusion Matrix)', fontsize=13, pad=10)
axes[0].set_xlabel('Предсказанные классы', fontsize=11)
axes[0].set_ylabel('Истинные классы', fontsize=11)

# 2. ROC-кривая
axes[1].plot(fpr, tpr, color='#1f77b4', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='grey', lw=1.5, linestyle='--')
axes[1].scatter([target_fpr], [recall_at_fixed_fpr], color='#d62728', s=100, zorder=5, 
               label=f'Recall@1%FPR = {recall_at_fixed_fpr:.2f}')
axes[1].set_xlim([-0.01, 1.01])
axes[1].set_ylim([-0.01, 1.01])
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('Кривая ROC', fontsize=13, pad=10)
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(True, alpha=0.3)

# 3. PR-кривая
pr_p, pr_r, _ = precision_recall_curve(test_true, test_probs)
axes[2].plot(pr_r, pr_p, color='#2ca02c', lw=2.5, label=f'PR curve (AUC = {pr_auc:.4f})')
test_p_at_opt = report['1']['precision']
test_r_at_opt = report['1']['recall']
axes[2].scatter([test_r_at_opt], [test_p_at_opt], color='#d62728', s=100, zorder=5,
               label=f'Порог F2 ({best_thr:.2f})')
axes[2].set_xlim([-0.01, 1.01])
axes[2].set_ylim([-0.01, 1.01])
axes[2].set_xlabel('Recall', fontsize=11)
axes[2].set_ylabel('Precision', fontsize=11)
axes[2].set_title('Кривая Precision-Recall', fontsize=13, pad=10)
axes[2].legend(loc='lower left', fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('='*85)
print('📊 РЕЗУЛЬТАТЫ ФИНАЛЬНОЙ МОДЕЛИ HST-UEBA')
print('='*85)
print(f"Precision: {report['1']['precision']:.4f} | Recall: {report['1']['recall']:.4f} | F1-score: {report['1']['f1-score']:.4f} | F2-score: {fbeta_score(test_true, final_test_preds, beta=2):.4f}")
print(f"Recall@1%FPR: {recall_at_fixed_fpr:.4f} (FPR порог: {thr_at_fixed_fpr:.4f}) | MTTD: {mttd_value:.2f} дней")
print('='*85)

## 7.1. Экспорт обученной модели и нормализационных параметров
Сохраняем веса модели, Z-параметры `tsle_mean` / `tsle_std` и оптимальный $F_2$-порог для интеграции в бэкенд.

In [ ]:
save_dict = {
    'model_state_dict': model.state_dict(),
    'vocab_size': VOCAB_SIZE,
    'window_size': WINDOW_SIZE,
    'best_threshold': best_thr,
    'best_pr_auc': best_overall_val_pr_auc,
    'tsle_mean': tsle_mean,
    'tsle_std': tsle_std,
    'global_mean': DEV_GLOBAL_MEAN.tolist(),
    'global_std': DEV_GLOBAL_STD.tolist(),
}
torch.save(save_dict, 'hierarchical_ueba_transformer.pth')
print('Модель и нормализационные метаданные успешно сохранены в файл: hierarchical_ueba_transformer.pth')

try:
    from google.colab import files
    files.download('hierarchical_ueba_transformer.pth')
    print('Автоматическое скачивание файла модели запущено в браузере.')
except ImportError:
    print('Автоматическое скачивание пропущено (запуск вне окружения Google Colab).')

## 7.2. Оценка качества модели по сценариям инсайдерских атак CERT r5.2
Для глубокого анализа качества модели в рамках дипломной работы проведем детальную оценку Recall по 5 основным сценариям инсайдерской активности, описанным в наборе данных CERT.

In [ ]:
from datetime import datetime
ANSWERS_DIR = r'D:\Политех\Мага\Дипломы\М\12841247\answers'
user_day_scenarios = {}

if os.path.exists(ANSWERS_DIR):
    scenarios = [d for d in os.listdir(ANSWERS_DIR) if d.startswith('r5.2')]
    for scen in scenarios:
        scen_path = os.path.join(ANSWERS_DIR, scen)
        # Читаем все csv файлы в подпапках
        for fname in os.listdir(scen_path):
            if fname.endswith('.csv'):
                fpath = os.path.join(scen_path, fname)
                try:
                    with open(fpath, 'r', encoding='utf-8') as f:
                        for row in csv.reader(f):
                            if len(row) >= 4:
                                event_id, date_str, u = row[1].strip(), row[2].strip(), row[3].strip()
                                if u:
                                    try:
                                        dt = datetime.strptime(date_str, '%m/%d/%Y %H:%M:%S')
                                        day_key = dt.strftime('%Y-%m-%d')
                                        user_day_scenarios[(u, day_key)] = scen
                                    except Exception:
                                        pass
                except Exception:
                    pass

scenario_stats = {}
if user_day_scenarios:
    for i in range(len(test_true)):
        if test_true[i] == 1.0:
            u = win_u[test_idx][i]
            last_day_dt = win_dates[test_idx][i][-1]
            last_day_str = pd.to_datetime(last_day_dt).strftime('%Y-%m-%d')
            scen = user_day_scenarios.get((u, last_day_str), 'unknown')
            
            if scen not in scenario_stats:
                scenario_stats[scen] = {'TP': 0, 'FN': 0}
            
            pred = (test_probs[i] >= best_thr)
            if pred:
                scenario_stats[scen]['TP'] += 1
            else:
                scenario_stats[scen]['FN'] += 1
                
    print('📊 МЕТРИКИ ВЫЯВЛЕНИЯ ПО СЦЕНАРИЯМ CERT R5.2 (ТЕСТОВАЯ ВЫБОРКА)')
    print('-' * 70)
    print(f"{'Сценарий':<35} | {'TP':<8} | {'FN':<8} | {'Recall':<10}")
    print('-' * 70)
    for scen, stats in sorted(scenario_stats.items()):
        tp = stats['TP']
        fn = stats['FN']
        total = tp + fn
        rec = tp / total if total > 0 else 0.0
        
        names = {
            'r5.2-1': 'r5.2-1 (хищение через USB)',
            'r5.2-2': 'r5.2-2 (ИТ-саботаж)',
            'r5.2-3': 'r5.2-3 (хищение IP)',
            'r5.2-4': 'r5.2-4 (шпионаж/email)',
            'r5.2-5': 'r5.2-5 (аномальный веб-серфинг)'
        }
        scen_name = names.get(scen, scen)
        print(f"{scen_name:<35} | {tp:<8d} | {fn:<8d} | {rec:<10.4f}")
    print('-' * 70)
else:
    print('Директория answers не найдена. Анализ Recall по сценариям невозможен.')
    print('Пожалуйста, укажите верный путь к ответам в переменной ANSWERS_DIR.')

## 8. Интерпретируемость (Двойной XAI: Сравнительный анализ Attention & Causal Occlusion)
Анализируем веса внимания трансформера для **3 примеров истинных атак (True Positives)** и **3 примеров ложных срабатываний (False Positives)**, а также проводим **анализ окклюзионной чувствительности (Occlusion Sensitivity Analysis)** для причинно-следственной верификации важности дней скользящего окна.

In [ ]:
model.eval()
tps_found = []
fps_found = []

def calculate_occlusion_importance(model_obj, x_tensor, h_tensor, wh_tensor, tsle_tensor, dow_tensor, dev_tensor, base_p):
    importances = []
    for day in range(WINDOW_SIZE):
        mx = x_tensor.clone()
        mh = h_tensor.clone()
        mwh = wh_tensor.clone()
        mtsle = tsle_tensor.clone()
        mdow = dow_tensor.clone()
        mdev = dev_tensor.clone()
        
        mx[0, day] = 0
        mh[0, day] = 0
        mwh[0, day] = 0.0
        mtsle[0, day] = 0.0
        mdev[0, day] = 0.0
        
        with torch.no_grad():
            lgs, _, _ = model_obj(mx, mh, mwh, mtsle, mdow, mdev, extract_attention=False)
            mp = torch.sigmoid(lgs).item()
        importances.append(base_p - mp)
    return np.array(importances)

with torch.no_grad():
    for batch_X, batch_h, batch_wh, batch_tsle, batch_dow, batch_dev, batch_y in test_loader:
        for i in range(len(batch_y)):
            x_i = batch_X[i:i+1].to(device)
            h_i = batch_h[i:i+1].to(device)
            wh_i = batch_wh[i:i+1].to(device)
            tsle_i = batch_tsle[i:i+1].to(device)
            dow_i = batch_dow[i:i+1].to(device)
            dev_i = batch_dev[i:i+1].to(device)
            
            logits, _, attn_weights = model(x_i, h_i, wh_i, tsle_i, dow_i, dev_i, extract_attention=True)
            prob = torch.sigmoid(logits).item()
            is_mal = (batch_y[i].item() == 1.0)
            pred = (prob >= best_thr)
            
            if is_mal and pred and len(tps_found) < 3:
                imp = calculate_occlusion_importance(model, x_i, h_i, wh_i, tsle_i, dow_i, dev_i, prob)
                tps_found.append((prob, attn_weights[0].cpu().numpy(), imp))
            elif not is_mal and pred and len(fps_found) < 3:
                imp = calculate_occlusion_importance(model, x_i, h_i, wh_i, tsle_i, dow_i, dev_i, prob)
                fps_found.append((prob, attn_weights[0].cpu().numpy(), imp))
                
        if len(tps_found) >= 3 and len(fps_found) >= 3:
            break

# 1. Отрисовка True Positives (Истинные угрозы)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx in range(3):
    if idx < len(tps_found):
        prob, attn, importances = tps_found[idx]
        sns.heatmap(attn, cmap='Reds', annot=True, fmt='.2f', ax=axes[0, idx], cbar=(idx == 2),
                    xticklabels=[f'Д{d+1}' for d in range(WINDOW_SIZE)],
                    yticklabels=[f'Д{d+1}' for d in range(WINDOW_SIZE)])
        axes[0, idx].set_title(f'TP #{idx+1} Attention (p={prob:.3f})', fontsize=12, pad=10)
        axes[0, idx].set_xlabel('Keys', fontsize=9)
        axes[0, idx].set_ylabel('Queries', fontsize=9)
        
        colors = ['#d62728' if imp > 0 else '#7f7f7f' for imp in importances]
        axes[1, idx].bar([f'День {d+1}' for d in range(WINDOW_SIZE)], importances, color=colors, edgecolor='black', alpha=0.85)
        axes[1, idx].set_title(f'TP #{idx+1} Causal Importance (Occlusion)', fontsize=12, pad=10)
        axes[1, idx].set_ylabel('Падение вероятности', fontsize=10)
        axes[1, idx].grid(True, alpha=0.3, linestyle='--')
    else:
        axes[0, idx].axis('off')
        axes[1, idx].axis('off')
plt.suptitle('Сравнительный анализ True Positives (Истинные угрозы): Attention vs Causal Occlusion', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

# 2. Отрисовка False Positives (Ложные тревоги)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx in range(3):
    if idx < len(fps_found):
        prob, attn, importances = fps_found[idx]
        sns.heatmap(attn, cmap='Oranges', annot=True, fmt='.2f', ax=axes[0, idx], cbar=(idx == 2),
                    xticklabels=[f'Д{d+1}' for d in range(WINDOW_SIZE)],
                    yticklabels=[f'Д{d+1}' for d in range(WINDOW_SIZE)])
        axes[0, idx].set_title(f'FP #{idx+1} Attention (p={prob:.3f})', fontsize=12, pad=10)
        axes[0, idx].set_xlabel('Keys', fontsize=9)
        axes[0, idx].set_ylabel('Queries', fontsize=9)
        
        colors = ['#ff7f0e' if imp > 0 else '#7f7f7f' for imp in importances]
        axes[1, idx].bar([f'День {d+1}' for d in range(WINDOW_SIZE)], importances, color=colors, edgecolor='black', alpha=0.85)
        axes[1, idx].set_title(f'FP #{idx+1} Causal Importance (Occlusion)', fontsize=12, pad=10)
        axes[1, idx].set_ylabel('Падение вероятности', fontsize=10)
        axes[1, idx].grid(True, alpha=0.3, linestyle='--')
    else:
        axes[0, idx].axis('off')
        axes[1, idx].axis('off')
plt.suptitle('Сравнительный анализ False Positives (Ложные тревоги): Attention vs Causal Occlusion', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()